# ChatGPT Archive Compiler — compile and render

This second notebook is the operational chronological compiler. It clones the selected private GitHub branch into ephemeral storage under **/content**, validates the complete compiler with synthetic data, then optionally runs the real ZIP → Archive IR → analysis → document → HTML/PDF workflow.

To keep the project compact, corpus analysis, explicit redaction, document construction, rendering, and integrity checks are combined here while remaining separate tested package APIs. Annual volumes are the default because they are substantially more reliable in Colab than one multi-thousand-page PDF. The single-volume option remains available. Notebook 02 builds the complementary thematic semantic atlas from the validated Archive IR.

Only the source export and generated outputs persist in Google Drive. Before running, add a Colab secret named **GITHUB_TOKEN** with read-only Contents access to **jcollins-bioinfo/chatgpt-archive-compiler** and enable Notebook access.

Privacy boundary: Colab runs on Google-hosted infrastructure. The real run is disabled until you provide an exact ZIP path and acknowledgment. Cells print aggregate counts, warning codes, commit identifiers, and output paths—not conversation titles, message text, node identifiers, warning locations, or source-derived exception text.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

In [ ]:
from pathlib import Path

REPO_BRANCH = "agent/rebuild-colab-workflow"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs" / "compiled"

BOOK_TITLE = "ChatGPT Conversation Archive"  # @param {type:"string"}
BOOK_SUBTITLE = ""  # @param {type:"string"}
BOOK_AUTHOR = ""  # @param {type:"string"}
VOLUME_MODE = "month"  # @param ["month", "year", "single"]
RENDER_PDF = True  # @param {type:"boolean"}
INCLUDE_REASONING_SUMMARIES = False  # @param {type:"boolean"}
INCLUDE_SYSTEM_MESSAGES = False  # @param {type:"boolean"}
INCLUDE_TOOL_MESSAGES = False  # @param {type:"boolean"}
MAX_CONVERSATIONS = 0  # @param {type:"integer"}

# Optional explicit regular-expression replacements. Patterns are never written to the manifest.
# Example: [{"pattern": r"person@example\.com", "replacement": "[EMAIL]"}]
CUSTOM_REDACTIONS = []

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Drive folder is missing: MyDrive/ChatGPT Data Export")
if not REPO_BRANCH.strip():
    raise ValueError("REPO_BRANCH must not be empty.")
if VOLUME_MODE not in {"month", "year", "single"}:
    raise ValueError("VOLUME_MODE must be 'month', 'year', or 'single'.")
if MAX_CONVERSATIONS < 0:
    raise ValueError("MAX_CONVERSATIONS must be zero or positive.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
print(f"Repository branch: {REPO_BRANCH}")
print(f"Volume mode: {VOLUME_MODE}")

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run one shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_github_token() -> str:
    """Read the private-repository token without exposing provider error text."""

    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab secret GITHUB_TOKEN is missing or Notebook access is disabled."
        ) from None
    if not token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is empty.")
    return token


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield a subprocess environment backed by a temporary askpass helper."""

    token = get_github_token()
    with tempfile.TemporaryDirectory(prefix="cac_git_auth_", dir="/content") as directory:
        helper = Path(directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment = os.environ.copy()
        environment.pop("GITHUB_TOKEN", None)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
                "GIT_TERMINAL_PROMPT": "0",
            }
        )
        try:
            yield environment
        finally:
            environment.pop("CAC_GIT_TOKEN", None)


def resolve_remote_commit() -> str:
    """Resolve the selected branch to exactly one 40-character Git commit SHA."""

    if (
        run_command(
            ["git", "check-ref-format", "--branch", REPO_BRANCH],
            capture_output=True,
            check=False,
        ).returncode
        != 0
    ):
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")

    expected_ref = f"refs/heads/{REPO_BRANCH}"
    with authenticated_git_environment() as environment:
        result = run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "ls-remote",
                "--exit-code",
                PUBLIC_REPO_URL,
                expected_ref,
            ],
            env=environment,
            capture_output=True,
        )
    matches = [
        fields[0]
        for line in result.stdout.splitlines()
        if len(fields := line.split()) == 2 and fields[1] == expected_ref
    ]
    if len(matches) != 1 or re.fullmatch(r"[0-9a-f]{40}", matches[0]) is None:
        raise RuntimeError("Selected branch did not resolve to exactly one Git commit.")
    return matches[0]

In [ ]:
import importlib

CHECKED_OUT_COMMIT = resolve_remote_commit()
REPO_DIR = Path(
    tempfile.mkdtemp(
        prefix=f"chatgpt-archive-compiler-{CHECKED_OUT_COMMIT[:12]}-",
        dir="/content",
    )
)
with authenticated_git_environment() as environment:
    run_command(
        [
            "git",
            "-c",
            "credential.helper=",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            "--no-tags",
            PUBLIC_REPO_URL,
            str(REPO_DIR),
        ],
        env=environment,
    )
run_command(["git", "checkout", "--detach", CHECKED_OUT_COMMIT], cwd=REPO_DIR)
actual_commit = run_command(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    capture_output=True,
).stdout.strip()
if actual_commit != CHECKED_OUT_COMMIT:
    raise RuntimeError("Ephemeral checkout did not resolve to the selected commit.")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-e",
        f"{REPO_DIR}[notebooks,pdf]",
    ]
)
repository_source = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(repository_source))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
compiler_module = importlib.import_module("chatgpt_archive_compiler.compiler")
exceptions_module = importlib.import_module("chatgpt_archive_compiler.exceptions")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")

CompileOptions = compiler_module.CompileOptions
RedactionRule = compiler_module.RedactionRule
VolumeMode = compiler_module.VolumeMode
analyze_archive = compiler_module.analyze_archive
compile_archive = compiler_module.compile_archive
ArchiveCompilationError = exceptions_module.ArchiveCompilationError
IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
summarize_archive = ingest_module.summarize_archive
write_archive_ir = serialization_module.write_archive_ir

imported_from = Path(archive_compiler.__file__).resolve()
if repository_source not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the ephemeral checkout.")
dirty = run_command(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=REPO_DIR,
    capture_output=True,
).stdout
if dirty:
    raise RuntimeError("Package installation unexpectedly changed the Git checkout.")

print(f"Ephemeral checkout: {REPO_DIR}")
print(f"Commit: {CHECKED_OUT_COMMIT}")
print(f"Package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Synthetic full-pipeline validation

This smoke test exercises the new reasoning classifications and creates both HTML and PDF in temporary Colab storage. It verifies that reasoning records are preserved but omitted from the default document, the final assistant answer remains, no warnings are produced, and PDF rendering works before any real export is read.


In [ ]:
import json
import zipfile

synthetic_conversation = {
    "id": "synthetic-compile",
    "title": "Synthetic compiler validation",
    "create_time": 1_735_689_600,
    "update_time": 1_735_689_800,
    "current_node": "assistant-final",
    "mapping": {
        "root": {"id": "root", "parent": None, "message": None},
        "user": {
            "id": "user",
            "parent": "root",
            "message": {
                "id": "message-user",
                "author": {"role": "user"},
                "create_time": 1_735_689_600,
                "content": {"content_type": "text", "parts": ["Synthetic question"]},
                "metadata": {},
            },
        },
        "thinking": {
            "id": "thinking",
            "parent": "user",
            "message": {
                "id": "message-thinking",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_650,
                "content": {
                    "content_type": "thoughts",
                    "thoughts": [{"summary": "Synthetic thinking trace"}],
                },
                "metadata": {},
            },
        },
        "recap": {
            "id": "recap",
            "parent": "thinking",
            "message": {
                "id": "message-recap",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_700,
                "content": {
                    "content_type": "reasoning_recap",
                    "content": "Synthetic reasoning recap",
                },
                "metadata": {},
            },
        },
        "assistant-final": {
            "id": "assistant-final",
            "parent": "recap",
            "message": {
                "id": "message-final",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_800,
                "content": {"content_type": "text", "parts": ["Synthetic final answer"]},
                "metadata": {},
            },
        },
    },
}

with tempfile.TemporaryDirectory(prefix="cac_compile_smoke_", dir="/content") as directory:
    temporary_path = Path(directory)
    synthetic_zip = temporary_path / "synthetic-export.zip"
    with zipfile.ZipFile(
        synthetic_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive_zip:
        archive_zip.writestr(
            "conversations.json",
            json.dumps([synthetic_conversation], allow_nan=False),
        )

    synthetic_archive = ingest_export_zip(
        synthetic_zip,
        limits=IngestLimits(),
        schema_mode=SchemaMode.STRICT,
    )
    synthetic_analysis = analyze_archive(synthetic_archive)
    synthetic_result = compile_archive(
        synthetic_archive,
        temporary_path / "compiled",
        options=CompileOptions(
            title="Synthetic Archive",
            volume_mode=VolumeMode.SINGLE,
            render_pdf=True,
        ),
    )
    synthetic_html = synthetic_result.volumes[0].html_path.read_text(encoding="utf-8")
    synthetic_pdf = synthetic_result.volumes[0].pdf_path
    assert synthetic_analysis.blocks_by_type["thinking_trace"] == 1
    assert synthetic_analysis.blocks_by_type["reasoning_summary"] == 1
    assert synthetic_archive.all_warnings == []
    assert "Synthetic final answer" in synthetic_html
    assert "Synthetic thinking trace" not in synthetic_html
    assert "Synthetic reasoning recap" not in synthetic_html
    assert synthetic_pdf is not None
    assert synthetic_pdf.read_bytes().startswith(b"%PDF")
    assert synthetic_result.manifest_path.is_file()

print("Synthetic ZIP-to-PDF validation passed.")
print(f"Structural messages: {synthetic_analysis.graph_message_count}")
print(f"Compiled volumes: {len(synthetic_result.volumes)}")

## Real archive compilation — disabled by default

Provide the exact existing export ZIP path in Drive, enable the Boolean control, and enter the acknowledgment exactly. This run re-ingests the ZIP so the newly supported **thoughts** and **reasoning_recap** records receive their intentional classifications. It writes a fresh Archive IR, privacy-safe structural analysis, monthly HTML/PDF volumes by default, an index, and a checksum manifest to one commit-stamped output directory. Monthly volumes keep each PDF layout job bounded; yearly and single-volume modes remain available.

The known exceptional source records—one missing mapping and one missing current node—are allowed. By default, any other warning code stops compilation after reporting only its code and count.


In [ ]:
from collections import Counter
from datetime import UTC, datetime

RUN_REAL_COMPILE = False  # @param {type:"boolean"}
REAL_EXPORT_ZIP = ""  # @param {type:"string"}
PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_ACKNOWLEDGEMENT = "I UNDERSTAND THIS RUNS IN GOOGLE COLAB"
FAIL_ON_UNEXPECTED_WARNINGS = True  # @param {type:"boolean"}
ALLOWED_WARNING_CODES = {"missing_mapping", "missing_current_node"}

if not RUN_REAL_COMPILE:
    print("Real archive compilation remains disabled.")
else:
    if PRIVACY_ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact privacy acknowledgment is required.")
    real_export_path = Path(REAL_EXPORT_ZIP).expanduser()
    if not real_export_path.is_file() or real_export_path.suffix.casefold() != ".zip":
        raise RuntimeError("REAL_EXPORT_ZIP must be an exact existing .zip file path.")

    try:
        real_archive = ingest_export_zip(
            real_export_path,
            limits=IngestLimits(max_warnings=100_000),
            schema_mode=SchemaMode.TOLERANT,
        )
    except Exception as exception:
        error_name = type(exception).__name__
        raise RuntimeError(
            f"Real-export ingestion failed safely ({error_name}); source-derived exception "
            "text was suppressed."
        ) from None

    real_summary = summarize_archive(real_archive)
    warning_codes = Counter(warning.code for warning in real_archive.all_warnings)
    unexpected_warning_codes = {
        code: count for code, count in warning_codes.items() if code not in ALLOWED_WARNING_CODES
    }
    if FAIL_ON_UNEXPECTED_WARNINGS and unexpected_warning_codes:
        raise RuntimeError(
            "Compilation stopped because unexpected structural warning codes remain: "
            f"{dict(sorted(unexpected_warning_codes.items()))}"
        )

    redaction_rules = tuple(RedactionRule.model_validate(item) for item in CUSTOM_REDACTIONS)
    compile_options = CompileOptions(
        title=BOOK_TITLE,
        subtitle=BOOK_SUBTITLE or None,
        author=BOOK_AUTHOR or None,
        volume_mode=VolumeMode(VOLUME_MODE),
        render_pdf=RENDER_PDF,
        include_reasoning_summaries=INCLUDE_REASONING_SUMMARIES,
        include_system_messages=INCLUDE_SYSTEM_MESSAGES,
        include_tool_messages=INCLUDE_TOOL_MESSAGES,
        max_conversations=MAX_CONVERSATIONS or None,
        redaction_rules=redaction_rules,
    )

    run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    real_output_directory = OUTPUT_ROOT / f"{run_id}-{CHECKED_OUT_COMMIT[:12]}"
    real_output_directory.mkdir(parents=True, exist_ok=False)
    archive_ir_path = write_archive_ir(
        real_archive,
        real_output_directory / "archive.ir.json",
    )
    try:
        compilation = compile_archive(
            real_archive,
            real_output_directory,
            options=compile_options,
        )
    except ArchiveCompilationError as exception:
        raise RuntimeError(str(exception)) from None
    except Exception as exception:
        error_name = type(exception).__name__
        raise RuntimeError(
            f"Document compilation failed safely ({error_name}); source-derived exception "
            "text was suppressed."
        ) from None

    pdf_count = sum(volume.pdf_path is not None for volume in compilation.volumes)
    print("Real archive compilation completed.")
    print(f"Conversations ingested: {real_summary.conversation_count}")
    print(f"Graph nodes: {real_summary.node_count}")
    print(f"Graph messages: {real_summary.message_count}")
    print(f"Current-path messages: {real_summary.current_path_message_count}")
    print(f"Warnings by code: {dict(sorted(warning_codes.items()))}")
    print(f"Conversations compiled: {compilation.selected_conversation_count}")
    print(f"Volumes: {len(compilation.volumes)}")
    print(f"PDF volumes: {pdf_count}")
    print(f"Archive IR: {archive_ir_path}")
    print(f"Compilation index: {compilation.index_path}")
    print(f"Integrity manifest: {compilation.manifest_path}")
    print(f"Output directory: {compilation.output_directory}")

## What persists

The repository checkout and synthetic artifacts remain under **/content** and disappear with the Colab runtime. The real Archive IR, analysis, HTML/PDF volumes, index, and manifest remain under **MyDrive/ChatGPT Data Export/outputs/compiled/<timestamp>-<commit>/**.

For thematic categorization and cross-conversation analysis, continue with **02_semantic_atlas_colab.ipynb**. It consumes the validated Archive IR produced by this workflow and writes a separate semantic-atlas edition.
